In [3]:
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.callbacks import EarlyStopping
import json

# Define constants
Input_Image = 256
Channels = 3
batch_size = 32
num_classes=4
input_shape = (Input_Image, Input_Image, Channels)
# Data augmentation for training set
train_data = tf.keras.preprocessing.image.ImageDataGenerator(
    rescale=1./255,
    horizontal_flip=True,
    rotation_range=10,
    shear_range=0.2,
    zoom_range=0.2,
    vertical_flip=True,
    width_shift_range=0.2,
    height_shift_range=0.2,
    fill_mode='nearest'
)

train_set = train_data.flow_from_directory(
    r"C:\Users\User\Desktop\MSERMSE_Temperature\New_folder\NEWWORK\train",
    target_size=(Input_Image, Input_Image),
    batch_size=batch_size,
    class_mode='categorical',
    shuffle=True
)

common_data = tf.keras.preprocessing.image.ImageDataGenerator(rescale=1./255)

test_set = common_data.flow_from_directory(
    r"C:\Users\User\Desktop\MSERMSE_Temperature\New_folder\NEWWORK\test",
    target_size=(Input_Image, Input_Image),
    batch_size=batch_size,
    class_mode='categorical'
)

val_set = common_data.flow_from_directory(
    r"C:\Users\User\Desktop\MSERMSE_Temperature\New_folder\NEWWORK\val",
    target_size=(Input_Image, Input_Image),
    batch_size=batch_size,
    class_mode='categorical'
)

# ICAI-V4 Architecture
def build_icai_v4(input_shape, num_classes):
    inputs = layers.Input(shape=input_shape)

    # Stem Layer
    x = layers.Conv2D(64, kernel_size=3, strides=2, activation='relu', padding='same')(inputs)
    x = layers.BatchNormalization()(x)

    # Coordinate Attention
    x_h = tf.reduce_mean(x, axis=1, keepdims=True)  # Reduce along height
    x_w = tf.reduce_mean(x, axis=2, keepdims=True)  # Reduce along width
    x_h = tf.tile(x_h, [1, x.shape[1], 1, 1])  # Broadcast along width
    x_w = tf.tile(x_w, [1, 1, x.shape[2], 1])  # Broadcast along height
    x = layers.Concatenate()([x_h, x_w])
    x = layers.Conv2D(32, kernel_size=1, activation='relu', padding='same')(x)
    x = layers.Conv2D(64, kernel_size=1, activation='sigmoid', padding='same')(x)

    # Inception Blocks
    def inception_block(x, filters):
        branch1 = layers.Conv2D(filters, kernel_size=1, activation='relu', padding='same')(x)
        branch2 = layers.Conv2D(filters, kernel_size=1, activation='relu', padding='same')(x)
        branch2 = layers.Conv2D(filters, kernel_size=3, padding='same', activation='relu')(branch2)
        branch3 = layers.Conv2D(filters, kernel_size=1, activation='relu', padding='same')(x)
        branch3 = layers.Conv2D(filters, kernel_size=5, padding='same', activation='relu')(branch3)
        return layers.Concatenate()([branch1, branch2, branch3])

    x = inception_block(x, 64)
    x = layers.MaxPooling2D(pool_size=(2, 2))(x)
    x = inception_block(x, 128)
    x = layers.MaxPooling2D(pool_size=(2, 2))(x)

    # Final Layers
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.2)(x)
    outputs = layers.Dense(num_classes, activation='softmax')(x)

    return models.Model(inputs, outputs)


# Instantiate the ICAI-V4 model
input_shape = (Input_Image, Input_Image, Channels)
num_classes = train_set.num_classes
icai_v4 = build_icai_v4(input_shape, num_classes)
icai_v4.summary()


# Compile the ICAI-V4 model
icai_v4.compile(
    optimizer='adam',
    loss=tf.keras.losses.CategoricalCrossentropy(from_logits=False),
    metrics=['accuracy']
)

# Callbacks
callbacks = [
    EarlyStopping(
        monitor="val_accuracy",
        patience=3,
        verbose=1,
        restore_best_weights=True,
    )
]

# Train the ICAI-V4 model
history = icai_v4.fit(
    train_set,
    epochs=30,
    validation_data=val_set,
    callbacks=callbacks
)

# Evaluate on test data
test_loss, test_accuracy = icai_v4.evaluate(test_set)
print(f"Test Accuracy: {test_accuracy * 100:.2f}%")

# Save training history to JSON file
history_path = 'icai_v4_training_history.json'
with open(history_path, 'w') as history_file:
    json.dump(history.history, history_file)

icai_v4.save("ICAI.h5")

Found 4745 images belonging to 4 classes.
Found 595 images belonging to 4 classes.
Found 592 images belonging to 4 classes.
Model: "model"
__________________________________________________________________________________________________
 Layer (type)                Output Shape                 Param #   Connected to                  
 input_3 (InputLayer)        [(None, 256, 256, 3)]        0         []                            
                                                                                                  
 conv2d_2 (Conv2D)           (None, 128, 128, 64)         1792      ['input_3[0][0]']             
                                                                                                  
 batch_normalization_2 (Bat  (None, 128, 128, 64)         256       ['conv2d_2[0][0]']            
 chNormalization)                                                                                 
                                                                     

c:\Users\User\Desktop\MSERMSE_Temperature\New_folder\NEWWORK\venv\lib\site-packages\keras\src\engine\training.py:3103: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(
